In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from itertools import product
from pathlib import Path

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

In [ ]:
datasets = ['ILI2019', 'NCHSdeaths', 'JHUcase', 'HHShosp', 'AUcase', 'CAcase', 'CANpositivity', 'CHNGinpatient', 'CHNGoutpatient', 'CPRadmissions', 'DVcli']
#datasets = ['DVcli']
horizons = [1, 2, 4, 7, 10, 14, 28]
model_names = [
    "repeat_last",
    "ARIMA",
    "Dlinear",
    "AGCRN",
    "ColaGNN",
    "DCRNN",
    "EpiGNN",
    "MTGNN",
    "STGCN",
    "GTS",
    "StemGNN",
    "STNorm",
    "EARTH",
    "GraphWaveNet",
]

metric_cols = [
    "mse",
    "mae",
    "rmse",
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
    "medse",
    "medae",
]

epi_options = [False, "ngm", "sir_incidence", "sir_percent"]
einn_options = [True, False]
filter_options = [True, False]
ti_options = [True, False]

In [ ]:
def eval_metrics(pred, target):
    mse = metrics.get_MSE(pred, target)
    mae = metrics.get_MAE(pred, target)
    rmse = metrics.get_RMSE(pred, target)
    mse_filtered = metrics.get_MSE_filtered(pred, target)
    rmse_filtered = metrics.get_RMSE_filtered(pred, target)
    mae_filtered = metrics.get_MAE_filtered(pred, target)
    medse = metrics.get_medSE(pred, target)
    medae = metrics.get_medAE(pred, target)
    return {
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "mse_filtered": mse_filtered,
        "mae_filtered": mae_filtered,
        "rmse_filtered": rmse_filtered,
        "medse": medse,
        "medae": medae,
    }

### ALL

In [ ]:
from pathlib import Path
from itertools import product
import math

import numpy as np
import pandas as pd
import torch

# -------------------------------------------------
# Extra metrics
# -------------------------------------------------
extra_metric_cols = ["avg_error", "win_rate"]
all_summary_metrics = metric_cols + extra_metric_cols

baseline_method = model_names[0]  # repeat_last


def read_prediction_file(file_path: Path, file_cache: dict) -> pd.DataFrame | None:
    """
    Read and lightly standardize one prediction file.
    Cached to avoid repeated disk reads.
    """
    if file_path in file_cache:
        return file_cache[file_path]

    if not file_path.exists():
        file_cache[file_path] = None
        return None

    df = pd.read_csv(file_path)

    if df.empty:
        print(f"Empty file: {file_path}")
        file_cache[file_path] = None
        return None

    # Parse timestamp for leave-one-month-out
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["month"] = df["timestamp"].dt.month

    # Parse other date-like columns if present, so merge keys are consistent
    for col in ["train_start", "train_end", "eval_start", "eval_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    file_cache[file_path] = df
    return df


def get_merge_keys(df_a: pd.DataFrame, df_b: pd.DataFrame) -> list[str]:
    """
    Keys used to align a method file with the repeat_last baseline file.
    """
    preferred_keys = [
        "retrain_id",
        "timestamp",
        "state_idx",
        "state",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
    ]
    return [col for col in preferred_keys if col in df_a.columns and col in df_b.columns]


def summarize_vals(vals: np.ndarray, clip_01: bool = False):
    """
    Same CI logic as before: mean +/- 1.96 * SEM.
    """
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = len(vals)

    if n == 0:
        return np.nan, np.nan, np.nan

    if n == 1:
        mean = vals[0]
        ci_low = np.nan
        ci_high = np.nan
    else:
        mean = vals.mean()
        sd = vals.std(ddof=1)
        sem = sd / math.sqrt(n)
        ci = 1.96 * sem
        ci_low = mean - ci
        ci_high = mean + ci

    if clip_01:
        mean = np.clip(mean, 0.0, 1.0) if np.isfinite(mean) else mean
        ci_low = np.clip(ci_low, 0.0, 1.0) if np.isfinite(ci_low) else ci_low
        ci_high = np.clip(ci_high, 0.0, 1.0) if np.isfinite(ci_high) else ci_high

    return mean, ci_low, ci_high

In [ ]:
from pathlib import Path
from itertools import product
import math

import numpy as np
import pandas as pd
import torch

# -------------------------------------------------
# Extra metrics
# -------------------------------------------------
extra_metric_cols = ["avg_error", "win_rate"]
all_summary_metrics = list(metric_cols) + extra_metric_cols

baseline_method = model_names[0]  # repeat_last

key_cols = ["dataset", "method", "horizon", "epi", "einn", "filter", "ti"]


def read_prediction_file(file_path: Path, file_cache: dict) -> pd.DataFrame | None:
    """
    Read and lightly standardize one prediction file.
    Cached to avoid repeated disk reads.
    """
    if file_path in file_cache:
        return file_cache[file_path]

    if not file_path.exists():
        file_cache[file_path] = None
        return None

    df = pd.read_csv(file_path)

    if df.empty:
        print(f"Empty file: {file_path}")
        file_cache[file_path] = None
        return None

    # Parse timestamp for leave-one-month-out
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["month"] = df["timestamp"].dt.month

    # Parse other date-like columns if present, so merge keys are consistent
    for col in ["train_start", "train_end", "eval_start", "eval_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    file_cache[file_path] = df
    return df


def get_merge_keys(df_a: pd.DataFrame, df_b: pd.DataFrame) -> list[str]:
    """
    Keys used to align a method file with the repeat_last baseline file.
    """
    preferred_keys = [
        "retrain_id",
        "timestamp",
        "state_idx",
        "state",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
    ]
    return [col for col in preferred_keys if col in df_a.columns and col in df_b.columns]


def summarize_vals(vals: np.ndarray, clip_01: bool = False):
    """
    Same CI logic as before: mean +/- 1.96 * SEM.
    """
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = len(vals)

    if n == 0:
        return np.nan, np.nan, np.nan

    if n == 1:
        mean = vals[0]
        ci_low = np.nan
        ci_high = np.nan
    else:
        mean = vals.mean()
        sd = vals.std(ddof=1)
        sem = sd / math.sqrt(n)
        ci = 1.96 * sem
        ci_low = mean - ci
        ci_high = mean + ci

    if clip_01:
        mean = np.clip(mean, 0.0, 1.0) if np.isfinite(mean) else mean
        ci_low = np.clip(ci_low, 0.0, 1.0) if np.isfinite(ci_low) else ci_low
        ci_high = np.clip(ci_high, 0.0, 1.0) if np.isfinite(ci_high) else ci_high

    return mean, ci_low, ci_high


def normalize_key_value(x):
    """
    Normalize key values so that CSV-loaded values compare cleanly to
    in-memory config values, e.g. True vs 'True', 1 vs '1'.
    """
    if pd.isna(x):
        return "<NA>"

    if isinstance(x, (bool, np.bool_)):
        return str(bool(x)).lower()

    if isinstance(x, (int, np.integer)):
        return str(int(x))

    if isinstance(x, (float, np.floating)):
        x = float(x)
        return str(int(x)) if x.is_integer() else repr(x)

    s = str(x).strip()

    if s.lower() in {"true", "false"}:
        return s.lower()

    try:
        f = float(s)
        return str(int(f)) if f.is_integer() else repr(f)
    except ValueError:
        return s


def make_key(row_or_dict) -> tuple:
    return tuple(normalize_key_value(row_or_dict[col]) for col in key_cols)


def load_existing_results(results_path: Path, dataset: str) -> pd.DataFrame:
    """
    Load existing whole_results_{dataset}.csv if present.
    Drops old pandas index columns like 'Unnamed: 0'.
    """
    if not results_path.exists():
        return pd.DataFrame(columns=key_cols)

    existing_df = pd.read_csv(results_path)

    unnamed_cols = [c for c in existing_df.columns if c.startswith("Unnamed:")]
    if unnamed_cols:
        existing_df = existing_df.drop(columns=unnamed_cols)

    if "dataset" not in existing_df.columns:
        existing_df["dataset"] = dataset

    return existing_df


def expected_grid_for_dataset(dataset: str) -> pd.DataFrame:
    """
    All rows that should exist for this dataset.
    """
    rows = []

    for method in model_names:
        for horizon in horizons:
            for epi, einn, filt, ti in product(
                epi_options, einn_options, filter_options, ti_options
            ):
                rows.append(
                    {
                        "dataset": dataset,
                        "method": method,
                        "horizon": horizon,
                        "epi": epi,
                        "einn": einn,
                        "filter": filt,
                        "ti": ti,
                    }
                )

    return pd.DataFrame(rows)


def compute_one_result_row(
    dataset,
    method,
    horizon,
    epi,
    einn,
    filt,
    ti,
    folder: Path,
    file_cache: dict,
):
    """
    Compute exactly one summary row.

    Returns None if the required method file or baseline file is missing,
    empty, or otherwise cannot be evaluated.
    """
    print(dataset, method, horizon, epi, einn, filt, ti)

    # -------------------------------------------------
    # Current method file
    # -------------------------------------------------
    file_name = (
        f"retrain_{dataset}_{method}__horizon={horizon}"
        f"__epi={epi}__einn={einn}__filter={filt}__ti={ti}.csv"
    )
    file_path = folder / file_name

    df_file = read_prediction_file(file_path, file_cache)
    if df_file is None:
        print(f"Missing or invalid method file: {file_path}")
        return None

    unique_months = sorted(df_file["month"].dropna().unique())
    if len(unique_months) == 0:
        print(f"No valid months in: {file_path}")
        return None

    # -------------------------------------------------
    # Matching repeat_last baseline file
    # -------------------------------------------------
    baseline_file_name = (
        f"retrain_{dataset}_{baseline_method}__horizon={horizon}"
        f"__epi={epi}__einn={einn}__filter={filt}__ti={ti}.csv"
    )
    baseline_file_path = folder / baseline_file_name

    df_baseline = read_prediction_file(baseline_file_path, file_cache)
    if df_baseline is None:
        print(f"Missing repeat_last baseline file: {baseline_file_path}")
        return None

    merge_keys = get_merge_keys(df_file, df_baseline)
    if len(merge_keys) == 0:
        print(f"No merge keys found for baseline comparison: {file_path}")
        return None

    metric_samples = {metric: [] for metric in all_summary_metrics}

    for month_out in unique_months:
        df_fold = df_file[df_file["month"] != month_out].copy()
        df_baseline_fold = df_baseline[df_baseline["month"] != month_out].copy()

        if df_fold.empty:
            print(f"Skipping {file_path}, leaving out month {month_out} gives empty fold")
            continue

        if df_baseline_fold.empty:
            print(
                f"Skipping baseline {baseline_file_path}, "
                f"leaving out month {month_out} gives empty fold"
            )
            continue

        # -----------------------------
        # Existing eval_metrics metrics
        # -----------------------------
        pred = torch.as_tensor(
            df_fold["y_pred"].to_numpy(), dtype=torch.float32
        )
        target = torch.as_tensor(
            df_fold["y_true"].to_numpy(), dtype=torch.float32
        )

        metrics_all = eval_metrics(pred, target)

        for metric in metric_cols:
            val = metrics_all[metric]
            val = val.item() if torch.is_tensor(val) else float(val)
            metric_samples[metric].append(val)

        # -----------------------------
        # Signed average error
        # -----------------------------
        avg_error = float((df_fold["y_pred"] - df_fold["y_true"]).mean())
        metric_samples["avg_error"].append(avg_error)

        # -----------------------------
        # Win rate vs repeat_last
        # ties count as 0.5
        # -----------------------------
        df_compare = df_fold.merge(
            df_baseline_fold[merge_keys + ["y_true", "y_pred"]],
            on=merge_keys,
            how="inner",
            suffixes=("_method", "_baseline"),
        )

        if df_compare.empty:
            print(
                f"No aligned rows for win_rate in {file_path} "
                f"(month_out={month_out})"
            )
        else:
            method_abs_err = np.abs(
                df_compare["y_pred_method"] - df_compare["y_true_method"]
            )
            baseline_abs_err = np.abs(
                df_compare["y_pred_baseline"] - df_compare["y_true_method"]
            )

            wins = (method_abs_err < baseline_abs_err).astype(float)
            ties = (method_abs_err == baseline_abs_err).astype(float)

            win_rate = float((wins + 0.5 * ties).mean())
            metric_samples["win_rate"].append(win_rate)

    row = {
        "dataset": dataset,
        "method": method,
        "horizon": horizon,
        "epi": epi,
        "einn": einn,
        "filter": filt,
        "ti": ti,
        "n_month_folds": len(unique_months),
    }

    for metric in all_summary_metrics:
        vals = np.asarray(metric_samples[metric], dtype=float)

        if metric == "win_rate":
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=True)
        else:
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=False)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci_low"] = ci_low
        row[f"{metric}_ci_high"] = ci_high

    return row


results_dfs = {}

for dataset in datasets:
    folder = Path(f"/net/dali/home/mscbio/rul98/EpiPatch/retrain0301_{dataset}")

    # Read existing cached results first
    results_path = Path(f"whole_results_submission_{dataset}.csv")
    out_path = Path(f"whole_results_submission2_{dataset}.csv")
    existing_df = load_existing_results(results_path, dataset)

    expected_df = expected_grid_for_dataset(dataset)

    existing_keys = set()
    if not existing_df.empty:
        missing_key_cols = [c for c in key_cols if c not in existing_df.columns]
        if missing_key_cols:
            raise ValueError(
                f"{results_path} is missing required key columns: {missing_key_cols}"
            )

        existing_keys = set(existing_df[key_cols].apply(make_key, axis=1))

    expected_df["_key"] = expected_df.apply(make_key, axis=1)
    missing_df = expected_df[~expected_df["_key"].isin(existing_keys)].drop(columns="_key")

    print(
        f"\n{dataset}: {len(existing_keys)} existing rows, "
        f"{len(missing_df)} missing rows to attempt.\n"
    )

    file_cache = {}
    new_rows = []

    for _, missing in missing_df.iterrows():
        row = compute_one_result_row(
            dataset=missing["dataset"],
            method=missing["method"],
            horizon=missing["horizon"],
            epi=missing["epi"],
            einn=missing["einn"],
            filt=missing["filter"],
            ti=missing["ti"],
            folder=folder,
            file_cache=file_cache,
        )

        if row is not None:
            new_rows.append(row)

    new_df = pd.DataFrame(new_rows)

    if existing_df.empty:
        updated_df = new_df.copy()
    elif new_df.empty:
        updated_df = existing_df.copy()
    else:
        updated_df = pd.concat([existing_df, new_df], ignore_index=True)

    if not updated_df.empty:
        updated_df = (
            updated_df
            .drop_duplicates(subset=key_cols, keep="last")
            .sort_values(["method", "horizon", "epi", "einn", "filter", "ti"])
            .reset_index(drop=True)
        )

    results_dfs[dataset] = updated_df

    # Write updated version back to the same cached results file
    updated_df.to_csv(out_path, index=False)

    print(
        f"{dataset}: wrote {len(updated_df)} total rows to {results_path}; "
        f"filled {len(new_df)} new rows.\n"
    )

print(results_dfs[datasets[0]])

### OUTBREAKS

In [ ]:
from pathlib import Path
from itertools import product
import math
from typing import Optional

import numpy as np
import pandas as pd
import torch


# -------------------------------------------------
# Consensus filtering helpers
# -------------------------------------------------
def filter_to_consensus_intervals(
    df_file: pd.DataFrame,
    intervals_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Keep only rows where:
      - df_file['state'] matches intervals_df['state_code']
      - df_file['timestamp'] is within [start, end] for at least one interval

    Also standardizes timestamp and adds:
      - month = timestamp.dt.month
    """
    df = df_file.copy()
    df["_row_id"] = np.arange(len(df))

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["state"] = df["state"].astype(str)

    intervals = intervals_df.copy()
    intervals["state_code"] = intervals["state_code"].astype(str)
    intervals["start"] = pd.to_datetime(intervals["start"])
    intervals["end"] = pd.to_datetime(intervals["end"])

    merged = df.merge(
        intervals,
        left_on="state",
        right_on="state_code",
        how="left",
    )

    in_interval = (
        merged["start"].notna()
        & (merged["timestamp"] >= merged["start"])
        & (merged["timestamp"] <= merged["end"])
    )

    matched_row_ids = merged.loc[in_interval, "_row_id"].unique()

    filtered = (
        df[df["_row_id"].isin(matched_row_ids)]
        .drop(columns="_row_id")
        .copy()
    )

    filtered["timestamp"] = pd.to_datetime(filtered["timestamp"])
    filtered["month"] = filtered["timestamp"].dt.month

    return filtered


def read_prediction_file(
    file_path: Path,
    file_cache: dict,
) -> Optional[pd.DataFrame]:
    """
    Read and lightly standardize one prediction file.
    Cached to avoid repeated disk reads.
    """
    if file_path in file_cache:
        return file_cache[file_path]

    if not file_path.exists():
        file_cache[file_path] = None
        return None

    df = pd.read_csv(file_path)

    if df.empty:
        print(f"Empty file: {file_path}")
        file_cache[file_path] = None
        return None

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    for col in ["train_start", "train_end", "eval_start", "eval_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    file_cache[file_path] = df
    return df


def get_merge_keys(df_a: pd.DataFrame, df_b: pd.DataFrame) -> list[str]:
    """
    Keys used to align a method file with the repeat_last baseline file.
    """
    preferred_keys = [
        "retrain_id",
        "timestamp",
        "state_idx",
        "state",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
    ]
    return [col for col in preferred_keys if col in df_a.columns and col in df_b.columns]


def summarize_vals(vals: np.ndarray, clip_01: bool = False):
    """
    Summarize fold-level values using mean +/- 1.96 * SEM.
    """
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = len(vals)

    if n == 0:
        return np.nan, np.nan, np.nan

    if n == 1:
        mean = vals[0]
        ci_low = np.nan
        ci_high = np.nan
    else:
        mean = vals.mean()
        sd = vals.std(ddof=1)
        sem = sd / math.sqrt(n)
        ci = 1.96 * sem
        ci_low = mean - ci
        ci_high = mean + ci

    if clip_01:
        mean = np.clip(mean, 0.0, 1.0) if np.isfinite(mean) else mean
        ci_low = np.clip(ci_low, 0.0, 1.0) if np.isfinite(ci_low) else ci_low
        ci_high = np.clip(ci_high, 0.0, 1.0) if np.isfinite(ci_high) else ci_high

    return mean, ci_low, ci_high


# -------------------------------------------------
# Incremental-cache helpers
# -------------------------------------------------
key_cols = ["dataset", "method", "horizon", "epi", "einn", "filter", "ti"]


def normalize_key_value(x):
    """
    Normalize key values so CSV-loaded values compare cleanly to in-memory
    config values, e.g. True vs 'True', 1 vs '1'.
    """
    if pd.isna(x):
        return "<NA>"

    if isinstance(x, (bool, np.bool_)):
        return str(bool(x)).lower()

    if isinstance(x, (int, np.integer)):
        return str(int(x))

    if isinstance(x, (float, np.floating)):
        x = float(x)
        return str(int(x)) if x.is_integer() else repr(x)

    s = str(x).strip()

    if s.lower() in {"true", "false"}:
        return s.lower()

    try:
        f = float(s)
        return str(int(f)) if f.is_integer() else repr(f)
    except ValueError:
        return s


def make_key(row_or_dict) -> tuple:
    return tuple(normalize_key_value(row_or_dict[col]) for col in key_cols)


def load_existing_results(results_path: Path, dataset: str) -> pd.DataFrame:
    """
    Load existing outbreak_results_{dataset}.csv if present.
    Drops old pandas index columns like 'Unnamed: 0'.
    """
    if not results_path.exists():
        return pd.DataFrame(columns=key_cols)

    existing_df = pd.read_csv(results_path)

    unnamed_cols = [c for c in existing_df.columns if c.startswith("Unnamed:")]
    if unnamed_cols:
        existing_df = existing_df.drop(columns=unnamed_cols)

    if "dataset" not in existing_df.columns:
        existing_df["dataset"] = dataset

    return existing_df


def expected_grid_for_dataset(dataset: str) -> pd.DataFrame:
    """
    All rows that should exist for this dataset.
    """
    rows = []

    for method in model_names:
        for horizon in horizons:
            for epi, einn, filt, ti in product(
                epi_options, einn_options, filter_options, ti_options
            ):
                rows.append(
                    {
                        "dataset": dataset,
                        "method": method,
                        "horizon": horizon,
                        "epi": epi,
                        "einn": einn,
                        "filter": filt,
                        "ti": ti,
                    }
                )

    return pd.DataFrame(rows)


# -------------------------------------------------
# Metric setup
# -------------------------------------------------
extra_metric_cols = ["avg_error", "win_rate"]
all_summary_metrics = list(metric_cols) + extra_metric_cols

baseline_method = model_names[0]  # repeat_last


# -------------------------------------------------
# Compute one missing consensus/outbreak result row
# -------------------------------------------------
def compute_one_consensus_result_row(
    dataset,
    method,
    horizon,
    epi,
    einn,
    filt,
    ti,
    folder: Path,
    intervals_df: pd.DataFrame,
    file_cache: dict,
):
    """
    Compute exactly one outbreak/consensus-filtered summary row.

    Returns None if the required method file or baseline file is missing,
    empty, outside consensus intervals, or otherwise cannot be evaluated.
    """
    print(dataset, method, horizon, epi, einn, filt, ti)

    # -----------------------------------------
    # Method file
    # -----------------------------------------
    file_name = (
        f"retrain_{dataset}_{method}__horizon={horizon}"
        f"__epi={epi}__einn={einn}__filter={filt}__ti={ti}.csv"
    )
    file_path = folder / file_name

    df_file_raw = read_prediction_file(file_path, file_cache)
    if df_file_raw is None:
        print(f"Missing or invalid method file: {file_path}")
        return None

    df_eval = filter_to_consensus_intervals(df_file_raw, intervals_df)

    if df_eval.empty:
        print(f"No rows in consensus intervals for: {file_path}")
        return None

    unique_months = sorted(df_eval["month"].dropna().unique())
    if len(unique_months) == 0:
        print(f"No valid months after consensus filtering for: {file_path}")
        return None

    # -----------------------------------------
    # Matching repeat_last baseline file
    # -----------------------------------------
    baseline_file_name = (
        f"retrain_{dataset}_{baseline_method}__horizon={horizon}"
        f"__epi={epi}__einn={einn}__filter={filt}__ti={ti}.csv"
    )
    baseline_file_path = folder / baseline_file_name

    df_baseline_raw = read_prediction_file(baseline_file_path, file_cache)
    if df_baseline_raw is None:
        print(f"Missing repeat_last baseline file: {baseline_file_path}")
        return None

    df_baseline_eval = filter_to_consensus_intervals(
        df_baseline_raw,
        intervals_df,
    )

    if df_baseline_eval.empty:
        print(f"No baseline rows in consensus intervals for: {baseline_file_path}")
        return None

    merge_keys = get_merge_keys(df_eval, df_baseline_eval)
    if len(merge_keys) == 0:
        print(f"No merge keys found for baseline comparison: {file_path}")
        return None

    metric_samples = {metric: [] for metric in all_summary_metrics}

    for month_out in unique_months:
        df_fold = df_eval[df_eval["month"] != month_out].copy()
        df_baseline_fold = df_baseline_eval[
            df_baseline_eval["month"] != month_out
        ].copy()

        if df_fold.empty:
            print(
                f"Skipping {file_path}, leaving out month {month_out} gives empty fold"
            )
            continue

        if df_baseline_fold.empty:
            print(
                f"Skipping baseline {baseline_file_path}, "
                f"leaving out month {month_out} gives empty fold"
            )
            continue

        # ---------------------------------
        # Existing eval_metrics metrics
        # ---------------------------------
        pred = torch.as_tensor(
            df_fold["y_pred"].to_numpy(),
            dtype=torch.float32,
        )
        target = torch.as_tensor(
            df_fold["y_true"].to_numpy(),
            dtype=torch.float32,
        )

        metrics_all = eval_metrics(pred, target)

        for metric in metric_cols:
            val = metrics_all[metric]
            val = val.item() if torch.is_tensor(val) else float(val)
            metric_samples[metric].append(val)

        # ---------------------------------
        # Average signed error
        # ---------------------------------
        avg_error = float((df_fold["y_pred"] - df_fold["y_true"]).mean())
        metric_samples["avg_error"].append(avg_error)

        # ---------------------------------
        # Win rate vs repeat_last
        # Preserves original behavior:
        # strict wins only, no half-credit for ties.
        # ---------------------------------
        df_compare = df_fold.merge(
            df_baseline_fold[merge_keys + ["y_true", "y_pred"]],
            on=merge_keys,
            how="inner",
            suffixes=("_method", "_baseline"),
        )

        if df_compare.empty:
            print(
                f"No aligned rows for win_rate in {file_path} "
                f"(month_out={month_out})"
            )
        else:
            method_abs_err = np.abs(
                df_compare["y_pred_method"] - df_compare["y_true_method"]
            )
            baseline_abs_err = np.abs(
                df_compare["y_pred_baseline"] - df_compare["y_true_method"]
            )

            win_rate = float((method_abs_err < baseline_abs_err).mean())
            metric_samples["win_rate"].append(win_rate)

    row = {
        "dataset": dataset,
        "method": method,
        "horizon": horizon,
        "epi": epi,
        "einn": einn,
        "filter": filt,
        "ti": ti,
        "n_month_folds": len(unique_months),
    }

    for metric in all_summary_metrics:
        vals = np.asarray(metric_samples[metric], dtype=float)

        if metric == "win_rate":
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=True)
        else:
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=False)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci_low"] = ci_low
        row[f"{metric}_ci_high"] = ci_high

    return row


# -------------------------------------------------
# Incremental main loop for outbreak/consensus results
# -------------------------------------------------
results_dfs_consensus = {}

for dataset in datasets:
    folder = Path(f"/net/dali/home/mscbio/rul98/EpiPatch/retrain0301_{dataset}")
    intervals_path = Path(f"{dataset}_consensus_intervals.csv")
    out_path = Path(f"outbreak_results_submission2_{dataset}.csv")
    results_path = Path(f"outbreak_results_submission_{dataset}.csv")

    if not intervals_path.exists():
        print(f"Missing consensus intervals file: {intervals_path}")
        continue

    intervals_df = pd.read_csv(intervals_path)

    # Read existing cached outbreak results first
    existing_df = load_existing_results(results_path, dataset)

    expected_df = expected_grid_for_dataset(dataset)

    existing_keys = set()
    if not existing_df.empty:
        missing_key_cols = [c for c in key_cols if c not in existing_df.columns]
        if missing_key_cols:
            raise ValueError(
                f"{results_path} is missing required key columns: {missing_key_cols}"
            )

        existing_keys = set(existing_df[key_cols].apply(make_key, axis=1))

    expected_df["_key"] = expected_df.apply(make_key, axis=1)
    missing_df = expected_df[~expected_df["_key"].isin(existing_keys)].drop(
        columns="_key"
    )

    print(
        f"\n{dataset}: {len(existing_keys)} existing outbreak rows, "
        f"{len(missing_df)} missing rows to attempt.\n"
    )

    file_cache = {}
    new_rows = []

    for _, missing in missing_df.iterrows():
        row = compute_one_consensus_result_row(
            dataset=missing["dataset"],
            method=missing["method"],
            horizon=missing["horizon"],
            epi=missing["epi"],
            einn=missing["einn"],
            filt=missing["filter"],
            ti=missing["ti"],
            folder=folder,
            intervals_df=intervals_df,
            file_cache=file_cache,
        )

        if row is not None:
            new_rows.append(row)

    new_df = pd.DataFrame(new_rows)

    if existing_df.empty:
        updated_df = new_df.copy()
    elif new_df.empty:
        updated_df = existing_df.copy()
    else:
        updated_df = pd.concat([existing_df, new_df], ignore_index=True)

    if not updated_df.empty:
        updated_df = (
            updated_df
            .drop_duplicates(subset=key_cols, keep="last")
            .sort_values(["method", "horizon", "epi", "einn", "filter", "ti"])
            .reset_index(drop=True)
        )

    results_dfs_consensus[dataset] = updated_df

    # Write updated version back to cached outbreak file
    updated_df.to_csv(out_path, index=False)

    print(
        f"{dataset}: wrote {len(updated_df)} total rows to {results_path}; "
        f"filled {len(new_df)} new rows.\n"
    )


if results_dfs_consensus:
    first_dataset = next(iter(results_dfs_consensus))
    print(results_dfs_consensus[first_dataset])
else:
    print("No datasets were processed.")

### NON OUTBREAKS

In [ ]:
from pathlib import Path
from itertools import product
import math
from typing import Optional

import numpy as np
import pandas as pd
import torch


# =========================================================
# Interval preprocessing / filtering
# =========================================================

def build_interval_lookup(
    intervals_df: pd.DataFrame,
) -> dict[str, list[tuple[np.datetime64, np.datetime64]]]:
    """
    Preprocess consensus intervals once per dataset.

    Returns:
      {
        state_code: [(start, end), ...],
        ...
      }
    """
    intervals = intervals_df.copy()
    intervals["state_code"] = intervals["state_code"].astype(str)
    intervals["start"] = pd.to_datetime(intervals["start"])
    intervals["end"] = pd.to_datetime(intervals["end"])

    intervals = intervals.dropna(subset=["state_code", "start", "end"]).copy()

    lookup: dict[str, list[tuple[np.datetime64, np.datetime64]]] = {}
    for state_code, g in intervals.groupby("state_code", sort=False):
        lookup[str(state_code)] = list(
            zip(
                g["start"].to_numpy(dtype="datetime64[ns]"),
                g["end"].to_numpy(dtype="datetime64[ns]"),
            )
        )

    return lookup


def filter_to_non_outbreak_periods_fast(
    df_file: pd.DataFrame,
    interval_lookup: dict[str, list[tuple[np.datetime64, np.datetime64]]],
) -> pd.DataFrame:
    """
    Keep only rows that are NOT inside any consensus interval for that state.

    Same intended output as the original non-outbreak filter, but avoids the
    row-expanding merge against intervals on every call.
    """
    if df_file.empty:
        out = df_file.copy()
        out["month"] = pd.Series(dtype="float64")
        return out

    timestamps = df_file["timestamp"].to_numpy(dtype="datetime64[ns]")
    states = df_file["state"].astype(str).to_numpy()

    keep_mask = np.ones(len(df_file), dtype=bool)

    # Process one state at a time
    for state in pd.unique(states):
        intervals = interval_lookup.get(state)
        if not intervals:
            continue

        state_mask = states == state
        idx = np.flatnonzero(state_mask)
        ts = timestamps[idx]

        in_any_interval = np.zeros(len(idx), dtype=bool)
        for start, end in intervals:
            in_any_interval |= (ts >= start) & (ts <= end)

        keep_mask[idx] = ~in_any_interval

    filtered = df_file.loc[keep_mask].copy()
    filtered["month"] = filtered["timestamp"].dt.month

    return filtered


# =========================================================
# File reading / caching
# =========================================================

def read_prediction_file(
    file_path: Path,
    file_cache: dict,
) -> Optional[pd.DataFrame]:
    """
    Read and lightly standardize one prediction file.
    Cached to avoid repeated disk reads.
    """
    if file_path in file_cache:
        return file_cache[file_path]

    if not file_path.exists():
        file_cache[file_path] = None
        return None

    df = pd.read_csv(file_path)

    if df.empty:
        print(f"Empty file: {file_path}")
        file_cache[file_path] = None
        return None

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    for col in ["train_start", "train_end", "eval_start", "eval_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    file_cache[file_path] = df

    return df


def get_filtered_non_outbreak_file(
    file_path: Path,
    file_cache: dict,
    filtered_cache: dict,
    interval_lookup: dict[str, list[tuple[np.datetime64, np.datetime64]]],
) -> Optional[pd.DataFrame]:
    """
    Return cached non-outbreak-filtered dataframe for a file.
    """
    if file_path in filtered_cache:
        return filtered_cache[file_path]

    df_raw = read_prediction_file(file_path, file_cache)
    if df_raw is None:
        filtered_cache[file_path] = None
        return None

    df_filtered = filter_to_non_outbreak_periods_fast(df_raw, interval_lookup)
    filtered_cache[file_path] = df_filtered

    return df_filtered


# =========================================================
# Metric / merge helpers
# =========================================================

def get_merge_keys(df_a: pd.DataFrame, df_b: pd.DataFrame) -> list[str]:
    """
    Keys used to align a method file with the repeat_last baseline file.
    """
    preferred_keys = [
        "retrain_id",
        "timestamp",
        "state_idx",
        "state",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
    ]

    return [col for col in preferred_keys if col in df_a.columns and col in df_b.columns]


def summarize_vals(vals: np.ndarray, clip_01: bool = False):
    """
    Summarize fold-level values using mean +/- 1.96 * SEM.
    """
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = len(vals)

    if n == 0:
        return np.nan, np.nan, np.nan

    if n == 1:
        mean = vals[0]
        ci_low = np.nan
        ci_high = np.nan
    else:
        mean = vals.mean()
        sd = vals.std(ddof=1)
        sem = sd / math.sqrt(n)
        ci = 1.96 * sem
        ci_low = mean - ci
        ci_high = mean + ci

    if clip_01:
        mean = np.clip(mean, 0.0, 1.0) if np.isfinite(mean) else mean
        ci_low = np.clip(ci_low, 0.0, 1.0) if np.isfinite(ci_low) else ci_low
        ci_high = np.clip(ci_high, 0.0, 1.0) if np.isfinite(ci_high) else ci_high

    return mean, ci_low, ci_high


# =========================================================
# Incremental-cache helpers
# =========================================================

key_cols = ["dataset", "method", "horizon", "epi", "einn", "filter", "ti"]


def normalize_key_value(x):
    """
    Normalize key values so CSV-loaded values compare cleanly to in-memory
    config values, e.g. True vs 'True', 1 vs '1'.
    """
    if pd.isna(x):
        return "<NA>"

    if isinstance(x, (bool, np.bool_)):
        return str(bool(x)).lower()

    if isinstance(x, (int, np.integer)):
        return str(int(x))

    if isinstance(x, (float, np.floating)):
        x = float(x)
        return str(int(x)) if x.is_integer() else repr(x)

    s = str(x).strip()

    if s.lower() in {"true", "false"}:
        return s.lower()

    try:
        f = float(s)
        return str(int(f)) if f.is_integer() else repr(f)
    except ValueError:
        return s


def make_key(row_or_dict) -> tuple:
    return tuple(normalize_key_value(row_or_dict[col]) for col in key_cols)


def load_existing_results(results_path: Path, dataset: str) -> pd.DataFrame:
    """
    Load existing non_outbreak_results_{dataset}.csv if present.
    Drops old pandas index columns like 'Unnamed: 0'.
    """
    if not results_path.exists():
        return pd.DataFrame(columns=key_cols)

    existing_df = pd.read_csv(results_path)

    unnamed_cols = [c for c in existing_df.columns if c.startswith("Unnamed:")]
    if unnamed_cols:
        existing_df = existing_df.drop(columns=unnamed_cols)

    if "dataset" not in existing_df.columns:
        existing_df["dataset"] = dataset

    return existing_df


def expected_grid_for_dataset(dataset: str) -> pd.DataFrame:
    """
    All rows that should exist for this dataset.
    """
    rows = []

    for method in model_names:
        for horizon in horizons:
            for epi, einn, filt, ti in product(
                epi_options,
                einn_options,
                filter_options,
                ti_options,
            ):
                rows.append(
                    {
                        "dataset": dataset,
                        "method": method,
                        "horizon": horizon,
                        "epi": epi,
                        "einn": einn,
                        "filter": filt,
                        "ti": ti,
                    }
                )

    return pd.DataFrame(rows)


# =========================================================
# Metric setup
# =========================================================

extra_metric_cols = ["avg_error", "win_rate"]
all_summary_metrics = list(metric_cols) + extra_metric_cols

baseline_method = model_names[0]  # repeat_last


# =========================================================
# Compute one missing non-outbreak result row
# =========================================================

def compute_one_non_outbreak_result_row(
    dataset,
    method,
    horizon,
    epi,
    einn,
    filt,
    ti,
    folder: Path,
    interval_lookup: dict[str, list[tuple[np.datetime64, np.datetime64]]],
    file_cache: dict,
    filtered_cache: dict,
):
    """
    Compute exactly one non-outbreak-filtered summary row.

    Returns None if the required method file or baseline file is missing,
    empty, outside non-outbreak periods, or otherwise cannot be evaluated.
    """
    print(dataset, method, horizon, epi, einn, filt, ti)

    # -----------------------------------------
    # Method file
    # -----------------------------------------
    file_name = (
        f"retrain_{dataset}_{method}__horizon={horizon}"
        f"__epi={epi}__einn={einn}__filter={filt}__ti={ti}.csv"
    )
    file_path = folder / file_name

    df_eval = get_filtered_non_outbreak_file(
        file_path=file_path,
        file_cache=file_cache,
        filtered_cache=filtered_cache,
        interval_lookup=interval_lookup,
    )

    if df_eval is None:
        print(f"Missing or invalid method file: {file_path}")
        return None

    if df_eval.empty:
        print(f"No rows in non-outbreak periods for: {file_path}")
        return None

    unique_months = sorted(df_eval["month"].dropna().unique())
    if len(unique_months) == 0:
        print(f"No valid months after non-outbreak filtering for: {file_path}")
        return None

    # -----------------------------------------
    # Matching repeat_last baseline file
    # -----------------------------------------
    baseline_file_name = (
        f"retrain_{dataset}_{baseline_method}__horizon={horizon}"
        f"__epi={epi}__einn={einn}__filter={filt}__ti={ti}.csv"
    )
    baseline_file_path = folder / baseline_file_name

    df_baseline_eval = get_filtered_non_outbreak_file(
        file_path=baseline_file_path,
        file_cache=file_cache,
        filtered_cache=filtered_cache,
        interval_lookup=interval_lookup,
    )

    if df_baseline_eval is None:
        print(f"Missing repeat_last baseline file: {baseline_file_path}")
        return None

    if df_baseline_eval.empty:
        print(f"No baseline rows in non-outbreak periods for: {baseline_file_path}")
        return None

    merge_keys = get_merge_keys(df_eval, df_baseline_eval)
    if len(merge_keys) == 0:
        print(f"No merge keys found for baseline comparison: {file_path}")
        return None

    metric_samples = {metric: [] for metric in all_summary_metrics}

    # -----------------------------------------
    # Precompute arrays for fold exclusions
    # -----------------------------------------
    method_month = df_eval["month"].to_numpy()
    baseline_month = df_baseline_eval["month"].to_numpy()

    pred_full_np = df_eval["y_pred"].to_numpy()
    target_full_np = df_eval["y_true"].to_numpy()

    pred_full_t = torch.as_tensor(pred_full_np, dtype=torch.float32)
    target_full_t = torch.as_tensor(target_full_np, dtype=torch.float32)

    method_keep_masks_np = {m: method_month != m for m in unique_months}
    method_keep_masks_t = {
        m: torch.as_tensor(mask, dtype=torch.bool)
        for m, mask in method_keep_masks_np.items()
    }
    baseline_keep_masks_np = {m: baseline_month != m for m in unique_months}

    # -----------------------------------------
    # Precompute method vs baseline alignment once
    # -----------------------------------------
    df_compare = df_eval[merge_keys + ["month", "y_true", "y_pred"]].merge(
        df_baseline_eval[merge_keys + ["y_true", "y_pred"]],
        on=merge_keys,
        how="inner",
        suffixes=("_method", "_baseline"),
    )

    compare_empty = df_compare.empty

    if not compare_empty:
        compare_month = df_compare["month"].to_numpy()

        method_abs_err_all = np.abs(
            df_compare["y_pred_method"].to_numpy()
            - df_compare["y_true_method"].to_numpy()
        )
        baseline_abs_err_all = np.abs(
            df_compare["y_pred_baseline"].to_numpy()
            - df_compare["y_true_method"].to_numpy()
        )

        strict_win_all = method_abs_err_all < baseline_abs_err_all

    # -----------------------------------------
    # Leave-one-month-out folds
    # -----------------------------------------
    for month_out in unique_months:
        method_mask_np = method_keep_masks_np[month_out]

        if not method_mask_np.any():
            print(
                f"Skipping {file_path}, leaving out month {month_out} gives empty fold"
            )
            continue

        baseline_mask_np = baseline_keep_masks_np[month_out]

        if not baseline_mask_np.any():
            print(
                f"Skipping baseline {baseline_file_path}, "
                f"leaving out month {month_out} gives empty fold"
            )
            continue

        method_mask_t = method_keep_masks_t[month_out]

        # ---------------------------------
        # Existing eval_metrics metrics
        # ---------------------------------
        pred = pred_full_t[method_mask_t]
        target = target_full_t[method_mask_t]

        metrics_all = eval_metrics(pred, target)

        for metric in metric_cols:
            val = metrics_all[metric]
            val = val.item() if torch.is_tensor(val) else float(val)
            metric_samples[metric].append(val)

        # ---------------------------------
        # Average signed error
        # ---------------------------------
        avg_error = float(
            (pred_full_np[method_mask_np] - target_full_np[method_mask_np]).mean()
        )
        metric_samples["avg_error"].append(avg_error)

        # ---------------------------------
        # Win rate vs repeat_last
        # Strict wins only: method error < baseline error
        # ---------------------------------
        if compare_empty:
            print(
                f"No aligned rows for win_rate in {file_path} "
                f"(month_out={month_out})"
            )
        else:
            compare_mask = compare_month != month_out

            if not compare_mask.any():
                print(
                    f"No aligned rows for win_rate in {file_path} "
                    f"(month_out={month_out})"
                )
            else:
                win_rate = float(strict_win_all[compare_mask].mean())
                metric_samples["win_rate"].append(win_rate)

    row = {
        "dataset": dataset,
        "method": method,
        "horizon": horizon,
        "epi": epi,
        "einn": einn,
        "filter": filt,
        "ti": ti,
        "n_month_folds": len(unique_months),
    }

    for metric in all_summary_metrics:
        vals = np.asarray(metric_samples[metric], dtype=float)

        if metric == "win_rate":
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=True)
        else:
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=False)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci_low"] = ci_low
        row[f"{metric}_ci_high"] = ci_high

    return row


# =========================================================
# Incremental main loop for non-outbreak results
# =========================================================

results_dfs_non_outbreak = {}

for dataset in datasets:
    folder = Path(f"/net/dali/home/mscbio/rul98/EpiPatch/retrain0301_{dataset}")
    intervals_path = Path(f"{dataset}_consensus_intervals.csv")
    results_path = Path(f"non_outbreak_results_submission_{dataset}.csv")
    out_path = Path(f"non_outbreak_results_submission2_{dataset}.csv")
    if not intervals_path.exists():
        print(f"Missing consensus intervals file: {intervals_path}")
        continue

    intervals_df = pd.read_csv(intervals_path)
    interval_lookup = build_interval_lookup(intervals_df)

    # Read existing cached non-outbreak results first
    existing_df = load_existing_results(results_path, dataset)

    expected_df = expected_grid_for_dataset(dataset)

    existing_keys = set()

    if not existing_df.empty:
        missing_key_cols = [c for c in key_cols if c not in existing_df.columns]
        if missing_key_cols:
            raise ValueError(
                f"{results_path} is missing required key columns: {missing_key_cols}"
            )

        existing_keys = set(existing_df[key_cols].apply(make_key, axis=1))

    expected_df["_key"] = expected_df.apply(make_key, axis=1)
    missing_df = expected_df[~expected_df["_key"].isin(existing_keys)].drop(
        columns="_key"
    )

    print(
        f"\n{dataset}: {len(existing_keys)} existing non-outbreak rows, "
        f"{len(missing_df)} missing rows to attempt.\n"
    )

    file_cache = {}
    filtered_cache = {}
    new_rows = []

    for _, missing in missing_df.iterrows():
        row = compute_one_non_outbreak_result_row(
            dataset=missing["dataset"],
            method=missing["method"],
            horizon=missing["horizon"],
            epi=missing["epi"],
            einn=missing["einn"],
            filt=missing["filter"],
            ti=missing["ti"],
            folder=folder,
            interval_lookup=interval_lookup,
            file_cache=file_cache,
            filtered_cache=filtered_cache,
        )

        if row is not None:
            new_rows.append(row)

    new_df = pd.DataFrame(new_rows)

    if existing_df.empty:
        updated_df = new_df.copy()
    elif new_df.empty:
        updated_df = existing_df.copy()
    else:
        updated_df = pd.concat([existing_df, new_df], ignore_index=True)

    if not updated_df.empty:
        updated_df = (
            updated_df
            .drop_duplicates(subset=key_cols, keep="last")
            .sort_values(["method", "horizon", "epi", "einn", "filter", "ti"])
            .reset_index(drop=True)
        )

    results_dfs_non_outbreak[dataset] = updated_df

    # Write updated version back to cached non-outbreak file
    updated_df.to_csv(out_path, index=False)

    print(
        f"{dataset}: wrote {len(updated_df)} total rows to {results_path}; "
        f"filled {len(new_df)} new rows.\n"
    )


if results_dfs_non_outbreak:
    first_dataset = next(iter(results_dfs_non_outbreak))
    print(results_dfs_non_outbreak[first_dataset])
else:
    print("No datasets were processed.")

### TUSOAI

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import torch

from EpiLearnSpatialTemporal import metrics


# -------------------------------------------------
# Config
# -------------------------------------------------

project_root = Path(".")

pred_root_gnn = project_root / "optaris_epipatch" / "gnn"

baseline_method = "repeat_last"
baseline_root_template = "/net/dali/home/mscbio/rul98/EpiPatch/retrain0301_{dataset}"

baseline_epi = False
baseline_einn = False
baseline_filter = False
baseline_ti = False

RESULT_SCHEMA_VERSION = "tusoai_all_false_baseline_strict_normalized_merge_v2"

simple_method_specs = [
    {
        "method": "tusoai",
        "source_label": "optaris_epipatch/gnn",
        "source_kind": "gnn",
    },
    {
        "method": "tusoai_simple_stgnn",
        "source_label": "TusoAI simple-STGNN",
        "source_kind": "simple_stgnn",
    },
]

extra_metric_cols = ["avg_error", "win_rate"]
all_summary_metrics = list(metric_cols) + extra_metric_cols

key_cols_simple = ["dataset", "method", "horizon"]


# -------------------------------------------------
# Generic helpers
# -------------------------------------------------

def summarize_vals(vals: np.ndarray, clip_01: bool = False):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = len(vals)

    if n == 0:
        return np.nan, np.nan, np.nan

    if n == 1:
        mean = vals[0]
        ci_low = np.nan
        ci_high = np.nan
    else:
        mean = vals.mean()
        sd = vals.std(ddof=1)
        sem = sd / math.sqrt(n)
        ci = 1.96 * sem
        ci_low = mean - ci
        ci_high = mean + ci

    if clip_01:
        mean = np.clip(mean, 0.0, 1.0) if np.isfinite(mean) else mean
        ci_low = np.clip(ci_low, 0.0, 1.0) if np.isfinite(ci_low) else ci_low
        ci_high = np.clip(ci_high, 0.0, 1.0) if np.isfinite(ci_high) else ci_high

    return mean, ci_low, ci_high


def read_prediction_file(file_path: Path, file_cache: dict) -> pd.DataFrame | None:
    if file_path in file_cache:
        return file_cache[file_path]

    if not file_path.exists():
        file_cache[file_path] = None
        return None

    df = pd.read_csv(file_path)

    if df.empty:
        print(f"Empty file: {file_path}")
        file_cache[file_path] = None
        return None

    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed:")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["month"] = df["timestamp"].dt.month

    for col in ["train_start", "train_end", "eval_start", "eval_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    file_cache[file_path] = df
    return df


def get_merge_keys(df_a: pd.DataFrame, df_b: pd.DataFrame) -> list[str]:
    preferred_keys = [
        
        "timestamp",
        "state_idx",
        "state",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
    ]
    return [col for col in preferred_keys if col in df_a.columns and col in df_b.columns]


def normalize_for_merge(df: pd.DataFrame, merge_keys: list[str]) -> pd.DataFrame:
    out = df.copy()

    date_cols = {
        "timestamp",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
    }

    int_like_cols = {
        "retrain_id",
        "state_idx",
    }

    for col in merge_keys:
        if col in date_cols:
            out[col] = (
                pd.to_datetime(out[col], errors="coerce")
                .dt.strftime("%Y-%m-%d")
            )
        elif col in int_like_cols:
            out[col] = (
                pd.to_numeric(out[col], errors="coerce")
                .astype("Int64")
                .astype(str)
            )
        elif col == "state":
            out[col] = out[col].astype(str).str.strip()
        else:
            out[col] = out[col].astype(str).str.strip()

    return out


# -------------------------------------------------
# Path helpers
# -------------------------------------------------

def resolve_prediction_path(dataset: str, horizon: int, source_kind: str) -> Path:
    if source_kind == "gnn":
        return pred_root_gnn / f"{dataset}_tusoai_horizon={horizon}.csv"

    if source_kind == "simple_stgnn":
        return (
            project_root
            / "simple_spatial_tid_regression_nospatial"
            / f"retrain_{dataset}_simple_stgnn__horizon={horizon}.csv"
        )

    raise ValueError(f"Unknown source_kind: {source_kind}")


def resolve_all_false_baseline_path(dataset: str, horizon: int) -> Path:
    folder = Path(baseline_root_template.format(dataset=dataset))

    file_name = (
        f"retrain_{dataset}_{baseline_method}__horizon={horizon}"
        f"__epi={baseline_epi}"
        f"__einn={baseline_einn}"
        f"__filter={baseline_filter}"
        f"__ti={baseline_ti}.csv"
    )

    return folder / file_name


# -------------------------------------------------
# Diagnostics
# -------------------------------------------------

def print_alignment_diagnostics(
    df_method_fold: pd.DataFrame,
    df_baseline_fold: pd.DataFrame,
    merge_keys: list[str],
    file_path: Path,
    baseline_file_path: Path,
    month_out,
):
    print("\n" + "=" * 100)
    print("STRICT NORMALIZED MERGE ALIGNMENT DIAGNOSTICS")
    print(f"Method file:   {file_path}")
    print(f"Baseline file: {baseline_file_path}")
    print(f"month_out:     {month_out}")
    print(f"Method rows:   {len(df_method_fold)}")
    print(f"Baseline rows: {len(df_baseline_fold)}")
    print(f"Merge keys:    {merge_keys}")

    method_norm = normalize_for_merge(df_method_fold, merge_keys)
    baseline_norm = normalize_for_merge(df_baseline_fold, merge_keys)

    print("\nProgressive unique-key merge counts:")
    progressive_keys = []

    for key in merge_keys:
        progressive_keys.append(key)

        left = method_norm[progressive_keys].drop_duplicates()
        right = baseline_norm[progressive_keys].drop_duplicates()
        inner = left.merge(right, on=progressive_keys, how="inner")

        print(
            f"  keys={progressive_keys}: "
            f"method_unique={len(left)}, "
            f"baseline_unique={len(right)}, "
            f"inner_unique={len(inner)}"
        )

    print("\nIndividual key overlap:")
    for key in merge_keys:
        method_vals = set(method_norm[key].dropna().astype(str).unique())
        baseline_vals = set(baseline_norm[key].dropna().astype(str).unique())
        overlap = method_vals & baseline_vals

        print(
            f"  {key}: "
            f"method_unique={len(method_vals)}, "
            f"baseline_unique={len(baseline_vals)}, "
            f"overlap_unique={len(overlap)}"
        )

        if len(overlap) == 0:
            print(f"    method examples:   {sorted(list(method_vals))[:5]}")
            print(f"    baseline examples: {sorted(list(baseline_vals))[:5]}")

    print("=" * 100 + "\n")


# -------------------------------------------------
# Metrics
# -------------------------------------------------

def compute_baseline_fold_metrics(
    df_baseline: pd.DataFrame,
    unique_months,
) -> dict:
    all_maes = []
    all_rmses = []

    for month_out in unique_months:
        df_fold = df_baseline[df_baseline["month"] != month_out].copy()

        if df_fold.empty:
            continue

        pred = torch.as_tensor(df_fold["y_pred"].to_numpy(), dtype=torch.float32)
        target = torch.as_tensor(df_fold["y_true"].to_numpy(), dtype=torch.float32)

        metrics_all = eval_metrics(pred, target)

        mae = metrics_all["mae"]
        mae = mae.item() if torch.is_tensor(mae) else float(mae)
        all_maes.append(mae)

        if "rmse" in metrics_all:
            rmse = metrics_all["rmse"]
            rmse = rmse.item() if torch.is_tensor(rmse) else float(rmse)
            all_rmses.append(rmse)

    all_maes = np.asarray(all_maes, dtype=float)
    all_maes = all_maes[np.isfinite(all_maes)]

    all_rmses = np.asarray(all_rmses, dtype=float)
    all_rmses = all_rmses[np.isfinite(all_rmses)]

    return {
        "baseline_mae_mean": float(all_maes.mean()) if len(all_maes) else np.nan,
        "baseline_rmse_mean": float(all_rmses.mean()) if len(all_rmses) else np.nan,
        "baseline_folds_used": int(len(all_maes)),
    }


def compute_row_level_win_rate_vs_baseline(
    df_method_fold: pd.DataFrame,
    df_baseline_fold: pd.DataFrame,
    file_path: Path,
    baseline_file_path: Path,
    month_out,
    print_diagnostics: bool = False,
) -> dict:
    merge_keys = get_merge_keys(df_method_fold, df_baseline_fold)

    if len(merge_keys) == 0:
        return {
            "win_rate": np.nan,
            "n_aligned_rows": 0,
            "merge_keys": "",
            "diagnostics_printed": False,
            "n_y_true_mismatches": np.nan,
        }

    method_norm = normalize_for_merge(df_method_fold, merge_keys)
    baseline_norm = normalize_for_merge(df_baseline_fold, merge_keys)

    df_compare = method_norm.merge(
        baseline_norm[merge_keys + ["y_true", "y_pred"]],
        on=merge_keys,
        how="inner",
        suffixes=("_method", "_baseline"),
    )

    diagnostics_printed = False

    if df_compare.empty:
        if print_diagnostics:
            print_alignment_diagnostics(
                df_method_fold=df_method_fold,
                df_baseline_fold=df_baseline_fold,
                merge_keys=merge_keys,
                file_path=file_path,
                baseline_file_path=baseline_file_path,
                month_out=month_out,
            )
            diagnostics_printed = True

        return {
            "win_rate": np.nan,
            "n_aligned_rows": 0,
            "merge_keys": ",".join(merge_keys),
            "diagnostics_printed": diagnostics_printed,
            "n_y_true_mismatches": np.nan,
        }

    y_true_mismatch = ~np.isclose(
        df_compare["y_true_method"].to_numpy(dtype=float),
        df_compare["y_true_baseline"].to_numpy(dtype=float),
        rtol=1e-5,
        atol=1e-8,
        equal_nan=True,
    )
    n_y_true_mismatches = int(y_true_mismatch.sum())

    if n_y_true_mismatches > 0:
        print(
            f"WARNING: {n_y_true_mismatches}/{len(df_compare)} aligned rows "
            f"have different y_true values for {file_path} vs {baseline_file_path}, "
            f"month_out={month_out}."
        )

    method_abs_err = np.abs(
        df_compare["y_pred_method"] - df_compare["y_true_method"]
    )
    baseline_abs_err = np.abs(
        df_compare["y_pred_baseline"] - df_compare["y_true_method"]
    )

    wins = (method_abs_err < baseline_abs_err).astype(float)
    ties = (method_abs_err == baseline_abs_err).astype(float)

    return {
        "win_rate": float((wins + 0.5 * ties).mean()),
        "n_aligned_rows": int(len(df_compare)),
        "merge_keys": ",".join(merge_keys),
        "diagnostics_printed": diagnostics_printed,
        "n_y_true_mismatches": n_y_true_mismatches,
    }


# -------------------------------------------------
# Incremental cache helpers
# -------------------------------------------------

def normalize_key_value(x):
    if pd.isna(x):
        return "<NA>"

    if isinstance(x, (bool, np.bool_)):
        return str(bool(x)).lower()

    if isinstance(x, (int, np.integer)):
        return str(int(x))

    if isinstance(x, (float, np.floating)):
        x = float(x)
        return str(int(x)) if x.is_integer() else repr(x)

    s = str(x).strip()

    try:
        f = float(s)
        return str(int(f)) if f.is_integer() else repr(f)
    except ValueError:
        return s.lower() if s.lower() in {"true", "false"} else s


def make_simple_key(row_or_dict) -> tuple:
    return tuple(normalize_key_value(row_or_dict[col]) for col in key_cols_simple)


def load_existing_simple_results(
    results_path: Path,
    dataset: str,
    method: str,
) -> pd.DataFrame:
    if not results_path.exists():
        return pd.DataFrame(columns=key_cols_simple)

    df = pd.read_csv(results_path)

    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed:")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    if "dataset" not in df.columns:
        df["dataset"] = dataset

    if "method" not in df.columns:
        df["method"] = method

    if "result_schema_version" not in df.columns:
        print(f"{results_path} has no result_schema_version; recomputing.")
        return pd.DataFrame(columns=key_cols_simple)

    current = df[df["result_schema_version"] == RESULT_SCHEMA_VERSION].copy()

    dropped = len(df) - len(current)
    if dropped:
        print(f"{results_path}: ignoring {dropped} cached rows from old schema.")

    return current


def expected_simple_grid_for_dataset(dataset: str, method: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "dataset": dataset,
                "method": method,
                "horizon": horizon,
            }
            for horizon in horizons
        ]
    )


# -------------------------------------------------
# Compute one row
# -------------------------------------------------

def compute_one_simple_result_row(
    dataset: str,
    method: str,
    horizon: int,
    source_label: str,
    source_kind: str,
    file_cache: dict,
):
    print(dataset, method, horizon)

    file_path = resolve_prediction_path(dataset, horizon, source_kind)
    df_file = read_prediction_file(file_path, file_cache)

    if df_file is None:
        print(f"Missing prediction file for {method}: {file_path}")
        return None

    baseline_file_path = resolve_all_false_baseline_path(dataset, horizon)
    df_baseline = read_prediction_file(baseline_file_path, file_cache)

    if df_baseline is None:
        print(f"Missing all-false repeat_last baseline file: {baseline_file_path}")
        return None

    unique_months = sorted(df_file["month"].dropna().unique())

    if len(unique_months) == 0:
        print(f"No valid months in: {file_path}")
        return None

    baseline_summary = compute_baseline_fold_metrics(df_baseline, unique_months)

    metric_samples = {metric: [] for metric in all_summary_metrics}

    win_rate_n_folds_used = 0
    win_rate_missing_folds = 0
    win_rate_aligned_rows_total = 0
    win_rate_y_true_mismatches_total = 0
    win_rate_merge_keys_seen = set()
    diagnostics_already_printed = False

    for month_out in unique_months:
        df_fold = df_file[df_file["month"] != month_out].copy()
        df_baseline_fold = df_baseline[df_baseline["month"] != month_out].copy()

        if df_fold.empty:
            print(f"Skipping {file_path}, month_out={month_out} gives empty method fold")
            continue

        if df_baseline_fold.empty:
            print(f"Skipping {baseline_file_path}, month_out={month_out} gives empty baseline fold")
            continue

        pred = torch.as_tensor(df_fold["y_pred"].to_numpy(), dtype=torch.float32)
        target = torch.as_tensor(df_fold["y_true"].to_numpy(), dtype=torch.float32)

        metrics_all = eval_metrics(pred, target)

        for metric in metric_cols:
            val = metrics_all[metric]
            val = val.item() if torch.is_tensor(val) else float(val)
            metric_samples[metric].append(val)

        avg_error = float((df_fold["y_pred"] - df_fold["y_true"]).mean())
        metric_samples["avg_error"].append(avg_error)

        win_info = compute_row_level_win_rate_vs_baseline(
            df_method_fold=df_fold,
            df_baseline_fold=df_baseline_fold,
            file_path=file_path,
            baseline_file_path=baseline_file_path,
            month_out=month_out,
            print_diagnostics=not diagnostics_already_printed,
        )

        if win_info["diagnostics_printed"]:
            diagnostics_already_printed = True

        if win_info["merge_keys"]:
            win_rate_merge_keys_seen.add(win_info["merge_keys"])

        win_rate_aligned_rows_total += win_info["n_aligned_rows"]

        if np.isfinite(win_info["n_y_true_mismatches"]):
            win_rate_y_true_mismatches_total += int(win_info["n_y_true_mismatches"])

        if np.isfinite(win_info["win_rate"]):
            metric_samples["win_rate"].append(win_info["win_rate"])
            win_rate_n_folds_used += 1
        else:
            win_rate_missing_folds += 1
            print(
                f"No row-level aligned baseline comparison for {file_path} "
                f"vs {baseline_file_path}, month_out={month_out}; omitted."
            )

    row = {
        "result_schema_version": RESULT_SCHEMA_VERSION,
        "dataset": dataset,
        "method": method,
        "horizon": horizon,
        "source_label": source_label,
        "source_kind": source_kind,
        "source_file": str(file_path),
        "baseline_method": baseline_method,
        "baseline_source": "all_false_repeat_last_prediction_file",
        "baseline_file": str(baseline_file_path),
        "baseline_epi": baseline_epi,
        "baseline_einn": baseline_einn,
        "baseline_filter": baseline_filter,
        "baseline_ti": baseline_ti,
        "baseline_mae_mean": baseline_summary["baseline_mae_mean"],
        "baseline_rmse_mean": baseline_summary["baseline_rmse_mean"],
        "baseline_folds_used": baseline_summary["baseline_folds_used"],
        "n_month_folds": len(unique_months),
        "win_rate_n_folds_used": win_rate_n_folds_used,
        "win_rate_missing_folds": win_rate_missing_folds,
        "win_rate_aligned_rows_total": win_rate_aligned_rows_total,
        "win_rate_y_true_mismatches_total": win_rate_y_true_mismatches_total,
        "win_rate_merge_keys": ";".join(sorted(win_rate_merge_keys_seen)),
    }

    for metric in all_summary_metrics:
        vals = np.asarray(metric_samples[metric], dtype=float)

        if metric == "win_rate":
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=True)
        else:
            mean, ci_low, ci_high = summarize_vals(vals, clip_01=False)

        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci_low"] = ci_low
        row[f"{metric}_ci_high"] = ci_high

    return row


# -------------------------------------------------
# Main loop
# -------------------------------------------------

results_simple = {}
all_results = []
file_cache = {}

for method_spec in simple_method_specs:
    method = method_spec["method"]
    source_label = method_spec["source_label"]
    source_kind = method_spec["source_kind"]

    results_simple[method] = {}

    for dataset in datasets:
        results_path = Path(f"whole_results_submission_{dataset}_{method}.csv")
        print(results_path)

        existing_df = load_existing_simple_results(results_path, dataset, method)
        expected_df = expected_simple_grid_for_dataset(dataset, method)

        existing_keys = set()

        if not existing_df.empty:
            missing_key_cols = [
                c for c in key_cols_simple
                if c not in existing_df.columns
            ]

            if missing_key_cols:
                raise ValueError(
                    f"{results_path} is missing required key columns: {missing_key_cols}"
                )

            existing_keys = set(
                existing_df[key_cols_simple].apply(make_simple_key, axis=1)
            )

        expected_df["_key"] = expected_df.apply(make_simple_key, axis=1)
        missing_df = expected_df[
            ~expected_df["_key"].isin(existing_keys)
        ].drop(columns="_key")

        print(
            f"\n{dataset} / {method}: "
            f"{len(existing_keys)} current-schema existing rows, "
            f"{len(missing_df)} missing rows to attempt.\n"
        )

        new_rows = []

        for _, missing in missing_df.iterrows():
            row = compute_one_simple_result_row(
                dataset=missing["dataset"],
                method=missing["method"],
                horizon=missing["horizon"],
                source_label=source_label,
                source_kind=source_kind,
                file_cache=file_cache,
            )

            if row is not None:
                new_rows.append(row)

        new_df = pd.DataFrame(new_rows)

        if existing_df.empty:
            updated_df = new_df.copy()
        elif new_df.empty:
            updated_df = existing_df.copy()
        else:
            updated_df = pd.concat([existing_df, new_df], ignore_index=True)

        if not updated_df.empty:
            updated_df = (
                updated_df
                .drop_duplicates(subset=key_cols_simple, keep="last")
                .sort_values(["horizon"])
                .reset_index(drop=True)
            )

        results_simple[method][dataset] = updated_df
        updated_df.to_csv(results_path, index=False)

        print(
            f"{dataset} / {method}: wrote {len(updated_df)} rows to {results_path}; "
            f"filled {len(new_df)} new rows.\n"
        )

        if not updated_df.empty:
            all_results.append(updated_df)


results_all = (
    pd.concat(all_results, ignore_index=True)
    if all_results
    else pd.DataFrame()
)

all_results_path = Path("whole_results_tusoai_variants_all_datasets.csv")
results_all.to_csv(all_results_path, index=False)

print(f"Wrote aggregate results to {all_results_path}")
print(results_all.head())
